### E.3 Lab 3 — Tiny VQC vs Logistic Regression

**Lab Access and Execution Guide**  

This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional and Student Volumes).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability.  

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_6_Quantum_Algorithms_for_Learning_Advanced_Challenge_Small_VQC_vs_Logistic_Regression.ipynb)


---


---
**Note for Lab Participants**

Each plot generated in this notebook is automatically saved as a `.png` file under:
Advanced_Labs/figures/


The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  
This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**: `/content/Advanced_Labs/figures/`  
- **Locally**: `Advanced_Labs/figures/` (next to your notebook)  
- These images are **not automatically added to GitHub** — commit/push them if you want them in the repo.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.

Once you implement code in this lab, you can include the `save_e_figure()` helper from the coded labs to automatically save plots.



---


**Chapter 7 — Quantum Machine Learning Architectures**

*Chapter 7* introduces variational quantum circuits (VQCs) as flexible machine learning models, emphasizing their architectural role in mapping classical data into quantum-enhanced feature spaces. It contrasts these quantum models with classical baselines, showing how variational layers exploit non-linear embeddings and entanglement to capture structure that linear models miss.

*Lab 3* operationalizes this architectural theme. By placing a simple variational quantum classifier beside Logistic Regression, learners see how architectural choices translate into performance differences. The activity grounds Chapter 7’s point: VQCs are not “magic,” but they can reveal patterns that linear baselines cannot capture, especially when data structure benefits from quantum-inspired feature maps.

**Advanced Lab 3 — Tiny VQC vs Logistic Regression**

This lab contrasts a linear classical baseline with a variational quantum model, showing how quantum-inspired feature mappings can capture non-linear structure in data. Participants progress from visualizing the raw dataset, through linear separation with Logistic Regression, to a trigonometric embedding proxy for quantum kernels, and finally to comparative diagnostics.

**Goal:** Build and evaluate a tiny VQC side by side with Logistic Regression. Observe how embedding strategies affect separability and classification accuracy, and use diagnostic plots to understand when quantum circuits begin to outperform linear baselines.
Cross-reference: Appendix E.2, Figures E.2.3a–c.


---

**Task 1 - Helper Utilities**

In [ ]:
# ---- Figure helper (robust; use in every coded lab) ----
import os
import matplotlib.pyplot as plt

def save_e_figure(fname: str,
                  subdir: str = "Advanced_Labs/figures",
                  fig=None):
    """
    Save the current or provided figure with a consistent path and filename.

    Args:
        fname (str): Canonical filename, e.g. "P2_AdvLab03_E.2.3a.png"
        subdir (str): Output subdirectory (default: "Advanced_Labs/figures")
        fig (matplotlib.figure.Figure): Optional figure object to save.
    """
    os.makedirs(subdir, exist_ok=True)

    if fig is None:
        fig = plt.gcf()

    # Ensure filename has .png extension
    if not fname.lower().endswith(".png"):
        fname = fname + ".png"

    outpath = os.path.join(subdir, fname)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160)
    print(f"Saved figure → {outpath}")


**Methodology Analysis**

This helper function standardizes how figures are saved across advanced labs. By enforcing a consistent subdirectory, file naming convention, and image resolution, it ensures that all outputs follow standard procedure. The function is flexible, allowing you to save the current figure or pass a specific figure object, while automatically appending the .png extension when missing.

**Participant Feedback**

When you run this cell, nothing visual will appear, but a confirmation message will print showing the saved figure’s path. This indicates the function is ready to use. If you later see a figure file saved with the correct P2_AdvLab03_E.2.x.png name, the helper is working as intended.

---
**Task 2 - Environment Setup**

In [ ]:
# === Environment Setup (CPU-only, safe to re-run) ===
# Purpose: Ensure required packages are present, import them, and print key versions.
import sys, subprocess, pkgutil, importlib

def _is_installed(pkg_name: str) -> bool:
    try:
        importlib.import_module(pkg_name)
        return True
    except Exception:
        return False

def ensure(pkg_pip_name: str, import_name: str | None = None):
    """
    Ensure a package is importable. If not, attempt a quiet pip install.
    pkg_pip_name: name to pass to `pip install`
    import_name: module name used with `importlib.import_module` (defaults to pkg_pip_name)
    """
    mod = import_name or pkg_pip_name
    if not _is_installed(mod):
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_pip_name])
        except Exception as e:
            print(f"⚠️ Could not install {pkg_pip_name}. Error: {e}")

# Core deps
ensure("qiskit")         # import name: qiskit
ensure("qiskit-aer", "qiskit_aer")
ensure("matplotlib")
ensure("numpy")
ensure("scikit-learn", "sklearn")
# Optional: PennyLane (disabled by default)
# ensure("pennylane")

# Imports (after ensure)
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
try:
    from qiskit_aer import Aer
    _aer_ok = True
except Exception as _e:
    print("⚠️ qiskit-aer not available, some simulators may be unavailable.")
    _aer_ok = False

from qiskit.quantum_info import Statevector, SparsePauliOp
try:
    from qiskit.visualization import plot_histogram
except Exception:
    plot_histogram = None
    print("⚠️ plot_histogram unavailable.")

# Reproducibility and non-interactive plotting
np.random.seed(42)
matplotlib.rcParams.update({"figure.dpi": 120})

# Version prints (expected by Participant Feedback)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Qiskit:", qiskit.__version__)
print("qiskit-aer:", "available" if _aer_ok else "missing")

# Optional: simple backend check
if _aer_ok:
    try:
        _backend = Aer.get_backend("aer_simulator")
        print("Aer backend:", _backend.name())
    except Exception as _e:
        print("⚠️ Aer installed but backend unresolved:", _e)


**Methodology Analysis (Preparation)**

This cell prepares a stable, CPU-only environment for the lab. It first checks for required packages and attempts a quiet install when a dependency is missing, then imports the modules in a controlled order. We fix a random seed for reproducibility and set plotting parameters for consistent visuals. Finally, we print the key library versions and report whether the Aer simulator backend is available, ensuring Participants can diagnose environment issues before running quantum or classical routines.

**Participant Feedback**

After running this cell, you should see lines printing the versions for Python, NumPy, Matplotlib, and Qiskit, plus a status line for qiskit-aer. If qiskit-aer is available, you will also see the Aer backend name (for example, aer_simulator). If any package reports as missing or a backend cannot be resolved, re-run the cell once. If the issue persists, check internet access for installations or install the missing package manually in your environment.


---

**Lab Overview - Tiny VQC vs Logistic Regression**

This lab contrasts a linear classical baseline with a variational quantum model, showing how quantum-inspired feature mappings can capture non-linear structure in data. Participants progress from visualizing the raw dataset, through linear separation with Logistic Regression, to a trigonometric embedding proxy for quantum kernels, and finally to comparative diagnostics.

**Challenge:**

Train a 2-qubit variational circuit classifier on a toy dataset and compare its decision boundary to a classical logistic regression.

**Implementation note:** 

This version uses Statevector.from_instruction (instead of shot-based sampling) for faster, deterministic training. Equivalent results can be obtained with the aer_simulator by adding qc.save_statevector().

**Expected Results**

* Figure E.2.3a (Training Data): The two clusters are clearly nonlinear in structure; no straight-line boundary can perfectly separate them.

* Figure E.2.3b (Logistic Regression): The linear decision boundary splits the plane with a straight cut, leading to misclassifications near the cluster overlap.

* Figure E.2.3c (Quantum-Inspired Feature Map): The trigonometric feature map bends the geometry, producing curved decision regions that better fit the data; classification error should drop compared to logistic regression.

* Figure E.2.3d (SVM, RBF Kernel): The nonlinear kernel allows the SVM to form smooth, flexible boundaries, often outperforming the linear baseline and sometimes matching or even surpassing the mapped classifier on this small dataset.


---

**Task 3 - Dataset Preparation**

In [ ]:
# chapter is 7 — Quantum Machine Learning Architectures (Advanced Part 2)
# Figures E.2.3a–d

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# ----- 1) Data: two clusters (nonlinear separation helpful) -----
rng = np.random.default_rng(7)
N = 80
X1 = np.c_[rng.normal(-1.0, 0.30, N//2), rng.normal( 0.0, 0.30, N//2)]
X2 = np.c_[rng.normal( 1.0, 0.30, N//2), rng.normal( 0.0, 0.30, N//2)]
X = np.vstack([X1, X2])
y = np.array([0]*(N//2) + [1]*(N//2))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)


**Methodology Analysis**

This cell synthesizes a two-cluster dataset to probe linear versus nonlinear separability. It draws two Gaussian clusters, stacks them into a single feature matrix X, and assigns class labels in y. A stratified train–test split preserves class balance across splits, and a fixed random seed ensures reproducibility. This setup provides a controlled playground for comparing linear baselines, quantum-inspired mappings, and nonlinear kernels.

**Participant Feedback**

After running, you should see arrays Xtr, Xte, ytr, and yte with the expected shapes, and class counts that remain balanced in both splits. If class counts look uneven or shapes are unexpected, confirm that stratify=y and the random_state are set as shown.

---
**Task 4 — Training data scatter (scatter = data structure)**

In [ ]:
# Task E.2.3a — Training data scatter
fig, ax = plt.subplots()
ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, s=25, edgecolor="k")
ax.set_title("Training data")
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")

# Save figure (one call only, no .png extension needed)
save_e_figure("P2_AdvLab03_E.2.3a")

plt.show()

**Figure E.2.3a. Training Data (Nonlinear Clusters).**
  
Training data scatter with class coloring. The layout shows cluster structure and regions of class overlap that will influence the learned decision boundaries for Tiny VQC and Logistic Regression.



**Expected Results**

You should see distinct clusters or curved manifolds if the data are non-linear, plus areas where points from different classes mix. Tighter clusters suggest higher confidence regions; mixed areas foreshadow ambiguous boundaries and potential errors.

**Technical Analysis (for the visual)**

This scatter visualizes separability in the raw feature space. If classes look roughly linearly separable, Logistic Regression should perform strongly. If boundaries appear curved or intertwined, the Tiny VQC’s expressive feature map may recover non-linear structure. Check scaling and class balance; extreme imbalance or skewed axes can hide or exaggerate separability.

**Intuition Sidebar**

Imagine two types of seeds scattered on soil. If the piles are well separated, a straight fence works. If they wind around each other, you need a bendable fence that can curve with the terrain.

---
**Task 5 — Logistic Regression (linear baseline)**
***Training Configuration***

In [ ]:
# Task E.2.3b — Logistic Regression (linear baseline)
from sklearn.linear_model import LogisticRegression  # ensure imported

# Train linear baseline
lr = LogisticRegression().fit(Xtr, ytr)

# Grid + decision function
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
    np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)
)
grid = np.c_[xx.ravel(), yy.ravel()]
Z = lr.predict(grid).reshape(xx.shape)

fig, ax = plt.subplots()
ax.contourf(xx, yy, Z, alpha=0.25)  # light decision regions
ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, s=25, edgecolor="k")  # training points
ax.set_title("Logistic Regression — Linear Decision Regions")
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")

# Save figure (one call only; no .png extension needed)
save_e_figure("P2_AdvLab03_E.2.3b")

plt.show()


**Figure E.2.3b. Logistic Regression decision regions in the raw feature space**.

The model produces a single linear separating boundary; points are training samples colored by class. This serves as the classical baseline for comparison with the Tiny VQC.

**Expected Results**

You should see a single straight boundary dividing the plane, with two colored regions corresponding to the predicted classes. If the data are roughly linearly separable, most points will lie on the correct side; if they curve or intertwine, misclassifications will appear in the overlap zones.

**Technical Analysis (for the visual)**

Logistic Regression learns a linear separator in the feature space. Its capacity is limited to affine boundaries, so performance degrades when classes follow curved manifolds. If the boundary looks oddly tilted or misplaced, verify feature scaling and inspect the regularization strength (C). Very small C can underfit, while very large C can overfit noisy edges.

**Intuition Sidebar**

This baseline is a straight fence. It is fast, stable, and often good enough. But when the terrain bends, a straight fence cannot hug the curves, so it will cut through the wrong places.

*Note:* This cell trains a Logistic Regression classifier on the raw feature space. It provides a transparent linear baseline with convex optimization and well-understood regularization. The resulting weight vector and intercept define a single affine decision boundary that will serve as a reference against more expressive approaches.

You should see the trained model available as lr. If you print coefficients, expect a small vector matching the feature dimension. If a convergence warning appears, consider increasing max_iter or checking feature scaling before proceeding.

---
**Task 6 —C — Quantum-Inspired feature map (proxy) — linear classifier**
***Training Configuration***

In [ ]:
# Task E.2.3c — Quantum-Inspired feature map (proxy) — linear classifier
from sklearn.linear_model import LogisticRegression  # ensure imported

# Simple "quantum-inspired" trigonometric feature map (proxy for a quantum feature map)
def phi(X):
    x1, x2 = X[:, 0], X[:, 1]
    return np.column_stack([
        np.cos(np.pi * x1), np.sin(np.pi * x1),
        np.cos(np.pi * x2), np.sin(np.pi * x2),
        np.cos(np.pi * x1 * x2), np.sin(np.pi * x1 * x2)
    ])

# Train linear classifier on mapped features (proxy for quantum kernel methods)
Xtr_phi = phi(Xtr); Xte_phi = phi(Xte)
clf_q = LogisticRegression(max_iter=2000).fit(Xtr_phi, ytr)

# Plot boundary in original space by classifying the mapped grid
grid_phi = phi(grid)
Zq = clf_q.predict(grid_phi).reshape(xx.shape)

fig, ax = plt.subplots()
ax.contourf(xx, yy, Zq, alpha=0.25)
ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, s=25, edgecolor="k")
ax.set_title("Quantum-Inspired Feature Map (Proxy) — Linear Classifier")
ax.set_xlabel("x₁"); ax.set_ylabel("x₂")

# Save figure (one call only)
save_e_figure("P2_AdvLab03_E.2.3c")

plt.show()


**Figure E.2.3c. Decision regions from a quantum-inspired feature map classifier.** 
 
The nonlinear embedding warps the input space so a linear head produces curved effective boundaries, serving as a proxy for quantum kernel methods.


**Expected Results**
Curved decision boundaries that usually reduce error versus raw-space Logistic Regression when classes are not linearly separable.

**Technical Analysis (for the visual)**
This embedding expands periodic interactions, so linear separability in feature space maps to nonlinear separability in the original plane. If boundaries look too straight, enrich the map (extra harmonics or interaction terms) or revisit scaling and regularization.

**Intuition Sidebar**
Do not force a straight line to fit a curve. Change the coordinates so the straight line bends in the right space, then map it back.

**Note**: This cell applies a trigonometric feature map φ(X) that expands inputs with sinusoidal and interaction terms, acting as a proxy for quantum kernel or embedding methods. A linear classifier trained on φ(X) learns a separator that is linear in the mapped space but nonlinear in the original coordinates, enabling curved boundaries without changing the head’s simplicity.

You should see the mapped arrays Xtr_phi, Xte_phi and a fitted classifier clf_q. If you print shapes, expect X*_phi to have more columns than X. If training is slow or warnings appear, modestly raise max_iter or check for duplicate or ill-conditioned features in the map.


---

===
**Task 7 — SVM (RBF kernel) — nonlinear decision regions/nonlinear classical reference**

In [ ]:
# Task E.2.3d — SVM (RBF kernel) — nonlinear decision regions

# Train SVM with a nonlinear (RBF) kernel on original features
from sklearn.svm import SVC  # ensure imported
svm = SVC(kernel="rbf", gamma="scale", C=1.0).fit(Xtr, ytr)

# Reuse the same grid from earlier (xx, yy, grid)
Z_svm = svm.predict(grid).reshape(xx.shape)

fig, ax = plt.subplots()
ax.contourf(xx, yy, Z_svm, alpha=0.25)
ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, s=25, edgecolor="k")
ax.set_title("SVM (RBF Kernel) — Nonlinear Decision Regions")
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")

# Save figure (one call only; no .png extension needed)
save_e_figure("P2_AdvLab03_E.2.3d")

plt.show()


**Figure E.2.3d. Nonlinear decision regions from an SVM with RBF kernel.**

The curved boundary closely follows the data manifold, providing a strong classical reference against which quantum and quantum-inspired models can be compared.


**Expected Results**

The SVM should yield a smoothly curved boundary wrapping around class clusters, outperforming the straight Logistic Regression separator and sometimes matching or exceeding the expressivity of the quantum-inspired map on small datasets.

**Technical Analysis (for the visual)**

The RBF kernel implicitly maps features into a higher-dimensional space, enabling flexible nonlinear separation. If the boundary looks overly tight (overfitting), reduce C or adjust γ. If it looks too loose, increase C to enforce stricter margins.

**Intuition Sidebar**

Think of the RBF kernel as drawing an elastic membrane around each class. The elasticity lets the boundary stretch and curve to hug the data, unlike a rigid straight line.

**Note:* This cell trains an SVM with an RBF kernel directly in the original feature space. The RBF kernel implicitly lifts data into a high-dimensional space where a maximal-margin separator can curve around clusters. The C parameter trades margin width for classification errors, while gamma="scale" adapts the kernel width to data variance.

You should obtain a trained svm model. If you inspect attributes like svm.support_.shape, expect a support-vector count less than the number of training points. If boundaries appear overly wiggly in later plots, reduce C or adjust gamma; if too coarse, increase C to tighten the margin.

---
**Conclusion**

This lab demonstrates how representation governs separability. Logistic Regression provides a transparent linear baseline. A feature-mapped linear classifier, serving as a proxy for a Tiny VQC, reshapes the data geometry to reveal low-error separators. The RBF kernel SVM establishes a strong classical reference for nonlinear capacity. Together, these models clarify where quantum-inspired or quantum embeddings can offer an advantage, and when established classical kernels already provide sufficient expressive power.

**Key Takeaways**

* Geometry defines separability: curved class structures defeat purely linear rules.

* Feature maps add flexibility: quantum or quantum-inspired embeddings reshape the space so a linear head can succeed.

* Classical kernels remain powerful: nonlinear kernels like the RBF-SVM provide a strong baseline for comparison.

* Adopt a diagnostic trio: test models in sequence — linear → mapped-linear → kernel — to benchmark both classical and quantum approaches.

**Congratulations**

Well done completing this advanced lab. You successfully compared classical and quantum-inspired approaches, examined how embeddings reshape decision boundaries, and evaluated performance against a robust nonlinear reference. These skills mirror the type of critical benchmarking expected in research and professional QAI practice.

### Appendix E → Appendix B Cross-Reference
See **Appendix B — Quick Self-Check, Chapter 7 — Quantum Machine Learning Architectures**:  
- Questions 2 and 3 (quantum vs classical decision boundaries).  
They correspond to the classification comparison shown in **E.3 Lab 3** (Figures E.3.3a–d).


---
**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.


---
